# Ward views

Ward-level maps built from the tables in `data/tables/` (`python scripts/build_tables.py all`) and the
ward boundaries in `data/raw/layers/` (`scripts/fetch_layers.py`). Each section shows the table it draws
next to the plot. Drawing helpers shared by every section: `scripts/ward_maps.py`. Every choice is a
named setting in the next cell.

**Status (2026-10-04):** §1–§3 built and reviewed; §4 not started.

| § | View | Reads | Status |
|---|---|---|---|
| 1 | Alder map (ward number + last name) and photo grid | `people`, `data/raw/photos/` (`scripts/fetch_photos.py`) | built |
| 2 | Tenure map (continuous color, capped at 22 years) + sorted bar chart | `people` | built |
| 3 | Absence map (% of council meetings absent, decision 1.4) + ranked table | `meetings`, `vote_events`, `member_votes`, `attendance` | built; color-scale cap still open |
| 4 | Split roll calls: table + one map colored by vote | `vote_events`, `member_votes` | *not built yet* |

Kernel: **Chicago Council**.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, "../scripts")
import ward_maps

# --- Settings (every choice made in this notebook) --------------------------

TABLES_DIR = Path("../data/tables")
PHOTOS_DIR = Path("../data/raw/photos")

# Ward shapes: "real" (2023 boundaries) or "tiles" (plan Phase 4, not built yet).
MAP_SHAPES = "real"

# Basemap under every map: "street", "gray" or "none" (ward_maps.BASEMAP_OPTIONS).
# User, 2026-10-04: gray. §1.3 draws all three side by side.
BASEMAP_CHOICE = "gray"

# Name shown under the ward number (user, 2026-10-04: last name only, keep "Jr.").
#   "last"      = text before the LAST comma of display_name, i.e. everything but the given name
#                 ("Martin, Matthew J." -> "Martin"; "Cardona, Jr., Felix" -> "Cardona, Jr.")
#   "as_stored" = display_name unchanged
ALDER_LABEL_NAME = "last"

FIGURE_SIZE_MAP = (12, 15)

# Label font on the §1.2 map (user, 2026-10-04: as large as possible). 9 is the largest size for
# which hand nudges were found that leave no two labels overlapping; §1.2's sweep shows counts by size.
MAP_LABEL_FONT_SIZE = 9

# Hand nudges for labels that would otherwise overlap at MAP_LABEL_FONT_SIZE on FIGURE_SIZE_MAP.
# {ward: (right, up)} in typographic points; negative = left / down. Wards not listed sit at
# their representative point. Set by eye on 2026-10-04, checked by ward_maps.label_overlaps.
LABEL_OFFSET_POINTS_BY_WARD = {
    "40": (-4, -22),   # Vasquez, Jr.: down into the lower part of the narrow ward, clear of 48 and 50
    "48": (28, 8),     # Manaa-Hoppenworth: right and up toward the lakefront (text extends over the lake)
    "33": (-12, 0),    # Rodriguez Sanchez: left, clear of 47
    "47": (8, 0),      # Martin: right, clear of 33
    "14": (0, 4),      # Gutierrez: up, clear of 15
    "15": (6, -12),    # Lopez: down and right, clear of 14
    "03": (0, -5),     # Dowell: down, clear of 4
    "04": (0, 5),      # Robinson: up, clear of 3
    "43": (10, 3),     # Knudsen: right and slightly up, clear of 32 and 2
    "35": (-5, 0),     # Quezada: left, clear of 32
    "50": (0, -3),     # Silverstein: down, clear of 49
    "49": (0, 2),      # Hadden: up, clear of 50
}

# Font sizes tried in §1.2's sweep (overlap count with and without the nudges above).
LABEL_FONT_SIZES_TO_SWEEP = [6, 7, 8, 9, 10, 11, 12]

# Photo grid display (files on disk are never changed).
#   PHOTO_DISPLAY_CROP: "square" = crop non-square photos to a square (user, 2026-10-04); "none" = as is
#   PHOTO_CROP_VERTICAL_ANCHOR: for a tall photo, "top" keeps the top (head room like the square
#   photos), "center" cuts top and bottom equally. §1.4 shows both for every non-square photo.
PHOTO_DISPLAY_CROP = "square"
PHOTO_CROP_VERTICAL_ANCHOR = "top"

# --- §2 tenure map ---
# Value shown: people.years_on_council (decision 1.2, computed in scripts/build_tables.py).
# Continuous color, no bins (user, 2026-10-04).
# Reversed viridis, so yellow = newest, dark purple = longest (user, 2026-10-04).
TENURE_COLORMAP = "viridis_r"
# Color scale bottom (user, 2026-10-04: lowest value):
#   "data_min" = lowest years among the 50 current alders (1.02 today)
#   "zero"     = 0
TENURE_COLOR_MIN_RULE = "data_min"
# Color scale top, in years (user, 2026-10-04: 22, so Harris's 19.8 differs a little from the top).
# Anyone above gets the top color (today: Beale 27.42, Mitts 26.73; their printed years are
# unchanged). None = highest value.
TENURE_COLOR_CAP_YEARS = 22
TENURE_FILL_OPACITY = 0.85           # user, 2026-10-04; basemap shows faintly through
TENURE_DECIMALS = 1                  # years printed on map labels and bars

# Label style on the tenure map (user, 2026-10-04: show both as options; §2.2 draws both):
#   "three_lines" = ward number / last name / years, all at TENURE_THREE_LINES_FONT_SIZE
#   "two_sizes"   = "ward last-name" at TENURE_NAME_FONT_SIZE, years below at TENURE_YEARS_FONT_SIZE
TENURE_LABEL_STYLE = "two_sizes"
TENURE_THREE_LINES_FONT_SIZE = 8     # largest size with nudges found and no overlaps (9 pt: 11 pairs with §1's nudges)
TENURE_NAME_FONT_SIZE = 8            # user, 2026-10-04 (was 7)
TENURE_YEARS_FONT_SIZE = 10          # user, 2026-10-04 (was 12)
# Hand nudges per style, {ward: (right, up)} points, for FIGURE_SIZE_MAP. Set by eye 2026-10-04 and
# checked by ward_maps.label_overlaps (§2.2). Most follow §1's nudges for the same crowded wards.
TENURE_LABEL_OFFSETS_BY_STYLE = {
    "three_lines": {
        "40": (-4, -22), "48": (28, 8),     # north lakefront: as §1
        "33": (-12, 0), "47": (8, 0),       # as §1
        "14": (0, 6), "15": (6, -16),       # as §1, a bit further apart (three lines are taller)
        "03": (0, -9), "04": (0, 8),        # as §1, further apart
        "43": (10, 3), "02": (0, -7),       # Knudsen as §1; Hopkins down, clear of 43
        "35": (-5, 0),                      # as §1
        "50": (0, -3), "49": (0, 2),        # as §1
        "46": (0, 7), "44": (6, 0),         # Clay up, Lawson right: clear of each other and of 32
        "29": (0, -6),                      # Taliaferro down, clear of 36
        "20": (4, 4), "16": (0, -3),        # Taylor up-right, Coleman down: clear of each other
    },
    "two_sizes": {                          # set for names 8 pt / years 10 pt
        "40": (-4, -22),                    # Vasquez, Jr.: as §1
        "48": (14, 8),                      # Manaa-Hoppenworth: right and up, less far east than §1 (user)
        "33": (-18, 0), "47": (8, 0), "46": (6, 0),   # long name lines in a row: spread left/right
        "30": (-8, 0), "31": (2, -6),       # Cruz left, Cardona down: clear of each other and of 35
        "35": (4, 8),                       # Quezada up-right, clear of 31 and 32
        "43": (12, 2),                      # Knudsen right, clear of 32 and 1
        "29": (-6, -4),                     # Taliaferro down-left, clear of 37
        "42": (4, 6),                       # Reilly up-right, clear of 34
        "14": (0, 6), "15": (6, -12),       # as §1
        "03": (0, -8), "04": (0, 8),        # as §1, further apart
        "50": (0, -4), "49": (0, 3),        # as §1, slightly further apart
    },
}
# Sizes tried in §2.2's sweep.
TENURE_THREE_LINES_SIZES_TO_SWEEP = [7, 8, 9]
TENURE_YEARS_SIZES_TO_SWEEP = [9, 10, 11, 12]   # names at TENURE_NAME_FONT_SIZE
# Bar chart size: 50 rows in 8.5 inches = about 11 pt per row, enough for the 8 pt name labels
# (user, 2026-10-04: less vertical spread; was 13 inches).
FIGURE_SIZE_BAR_CHART = (8, 8.5)
TENURE_BAR_HEIGHT = 0.75              # fraction of each row the bar fills (matplotlib default 0.8)
TENURE_BAR_FONT_SIZE = 8              # name labels and printed years
FIGURE_SIZE_BASEMAP_COMPARISON = (24, 11)

# --- §3 absence map ---
# Decision 1.4 (sessions/2026-10-04.md; same rule as tables_review §1.4d / §2.6): an alder is absent
# from a council meeting if their vote is "Absent" on MORE THAN this share of that meeting's roll-call
# items (exactly 50% counts as present). Not Voting is not Absent (decision 1.3).
ABSENT_MEETING_SHARE_THRESHOLD = 0.5
ABSENCE_MEETING_BODY = "City Council"        # meetings.body of the meetings counted
ABSENCE_ROSTER_KINDS = ["roll call"]         # vote_events.roster_kind counted (decision: metrics use roll call only)
# Events left out of the absence computation, {event_id: reason} (user, 2026-10-04). §3.1 shows the evidence.
EXCLUDE_EVENTS_FROM_ABSENCE = {
    "4016E3F7-F562-EE11-BE6E-001DD80523DD_49707A14-6968-EE11-9AE6-001DD80971B2":
        "R2023-0005012, 2023-10-11: only roll-call event at that meeting; its roster is the previous council's "
        "(Burke, Austin, Tunney, ...), marking Curtis Absent while the meeting's attendance shows all 50 Present",
}
# Denominator (user, 2026-10-04: meetings on roster):
#   "meetings_on_roster" = council meetings where the alder is on at least one counted roll-call roster
#                          (i.e. while seated; Burnett 18, Quezada 24, others 63 today)
#   "all_meetings"       = every council meeting with a counted roll-call event, for everyone
ABSENCE_DENOMINATOR = "meetings_on_roster"
# Colormap. §3.4 draws each option. User, 2026-10-04: "viridis" (not reversed), so yellow = most
# absent and dark purple = never absent.
ABSENCE_COLORMAP_OPTIONS = ["viridis", "viridis_r", "Reds"]
ABSENCE_COLORMAP = "viridis"
ABSENCE_COLOR_MIN_PERCENT = 0               # 0 = never absent
ABSENCE_COLOR_CAP_PERCENT = None            # None = highest value
ABSENCE_FILL_OPACITY = 0.85                 # same as TENURE_FILL_OPACITY
ABSENCE_PERCENT_DECIMALS = 1
# Value line under "ward name" (user, 2026-10-04: percent as the default for now; §3.3 draws both):
#   "both"    = "12.7% (8/63)"   (absent meetings / meetings in denominator)
#   "percent" = "12.7%"
ABSENCE_LABEL_VALUE = "percent"
ABSENCE_NAME_FONT_SIZE = 8                  # same sizes as the tenure "two_sizes" labels
ABSENCE_VALUE_FONT_SIZE = 10
# Hand nudges per label value, {ward: (right, up)} points, for FIGURE_SIZE_MAP. Checked in §3.3.
ABSENCE_LABEL_OFFSETS_BY_VALUE = {
    # Same width class as the tenure "two_sizes" labels: their nudges give no overlaps.
    "percent": TENURE_LABEL_OFFSETS_BY_STYLE["two_sizes"],
    # "12.7% (8/63)" is about 3x wider: 34 wards nudged, set by eye 2026-10-04, mostly sideways
    # apart from the neighbor they hit.
    "both": {
        "40": (-4, -22), "48": (14, 8), "49": (0, 3), "50": (0, -4),          # north lakefront: as tenure
        "33": (-30, 4), "47": (6, -8), "46": (16, 9), "44": (4, -10),         # 33/47/46/44 row
        "30": (-16, 4), "31": (4, -10), "35": (8, 10),                        # 30/31/35
        "36": (-10, 4), "26": (2, 2), "29": (-20, -4), "37": (6, 0),          # 36/26, 29/37
        "01": (2, 8), "02": (8, -10), "43": (20, 2), "42": (4, 6),            # downtown / near north
        "24": (-8, 6), "25": (8, -4), "22": (-12, 0), "12": (8, 0),           # 24/25, 22/12
        "04": (6, 8), "11": (-10, 0), "03": (0, -8),                          # 4/11/3
        "14": (0, 6), "15": (4, -8), "23": (-6, -12), "16": (14, -4),         # 14/15/23/16
        "08": (-20, 0), "07": (12, -10), "19": (-14, 0), "21": (10, 0),       # 8/7, 19/21
    },
}
FIGURE_SIZE_COLORMAP_COMPARISON = (24, 10)
PHOTO_GRID_COLUMNS = 10

pd.set_option("display.max_rows", 200)

## 1. Alder map

### 1.1 Current alder per ward — the table the map draws

Current alder = `people` row with `is_active` = True and a ward number in `elms_ward` (decision 1.1:
eLMS `/person`). The same rule picks whose photo `scripts/fetch_photos.py` downloads. Clerk and Mayor
rows are active but have no ward and are left off the map; they are listed below so nothing is
hidden.

In [ ]:
people_table = pd.read_csv(TABLES_DIR / "people.csv", dtype={"elms_ward": str})
has_ward_number = people_table.elms_ward.str.fullmatch(r"\d+", na=False)
current_alders = people_table[people_table.is_active & has_ward_number].copy()

print("active people without a ward number (not on the map):")
display(people_table[people_table.is_active & ~has_ward_number][["person_id", "display_name", "elms_ward"]])

# Label name, per ALDER_LABEL_NAME.
if ALDER_LABEL_NAME == "last":
    current_alders["label_name"] = current_alders.display_name.str.rsplit(",", n=1).str[0].str.strip()
elif ALDER_LABEL_NAME == "as_stored":
    current_alders["label_name"] = current_alders.display_name
else:
    raise ValueError(ALDER_LABEL_NAME)
current_alders["comma_count"] = current_alders.display_name.str.count(",")

print(f"current alders: {len(current_alders)}; distinct wards: {current_alders.elms_ward.nunique()}")
print("names whose display_name does not have exactly one comma (check their label):")
display(current_alders[current_alders.comma_count != 1][["elms_ward", "display_name", "label_name"]])

In [ ]:
# Join to ward shapes. Ward key: people.elms_ward "01" <-> layer ward "1", padded in ward_maps.load_ward_shapes.
ward_shapes = ward_maps.load_ward_shapes(MAP_SHAPES)
alder_map_table = ward_shapes.merge(current_alders.rename(columns={"elms_ward": "ward"}),
                                    on="ward", how="outer", indicator=True)
print("join result (both = ward shape and alder matched):", alder_map_table._merge.value_counts().to_dict())
assert (alder_map_table._merge == "both").all() and len(alder_map_table) == 50

alder_map_table = alder_map_table.sort_values("ward").reset_index(drop=True)
alder_map_table["map_label"] = alder_map_table.ward.str.lstrip("0") + "\n" + alder_map_table.label_name
display(alder_map_table[["ward", "ward_raw", "person_id", "display_name", "label_name"]])

### 1.2 Map — ward number and alder name

Labels sit at each ward's `representative_point()` (a point guaranteed inside the shape, which is
not always the visual center), moved by `LABEL_OFFSET_POINTS_BY_WARD` where listed. Font:
`MAP_LABEL_FONT_SIZE`. Basemap: `BASEMAP_CHOICE`. Some long names extend past their ward's outline.
After the map: the list of overlapping label pairs (expected: none), then a font-size sweep.

In [ ]:
map_label_by_ward = dict(zip(alder_map_table.ward, alder_map_table.map_label))

figure, axis = plt.subplots(figsize=FIGURE_SIZE_MAP)
ward_maps.draw_ward_outlines(axis, alder_map_table)
annotation_by_ward = ward_maps.draw_ward_labels(axis, alder_map_table, map_label_by_ward, font_size=MAP_LABEL_FONT_SIZE,
                                                offset_points_by_ward=LABEL_OFFSET_POINTS_BY_WARD)
ward_maps.add_basemap(axis, BASEMAP_CHOICE)
axis.set_title(f"Chicago City Council: ward and current alder (people.csv, eLMS /person) — basemap: {BASEMAP_CHOICE}")
axis.set_axis_off()
figure.canvas.draw()
map_label_overlaps = ward_maps.label_overlaps(figure, annotation_by_ward)
plt.show()

print(f"overlapping label pairs on this map: {len(map_label_overlaps)}")
display(pd.DataFrame(map_label_overlaps, columns=["ward_a", "ward_b", "overlap_width_px", "overlap_height_px"]))

In [ ]:
# Font-size sweep: overlapping label pairs at each size, without and with LABEL_OFFSET_POINTS_BY_WARD
# (same figure size, no basemap; the basemap does not move labels). The nudges were set for
# MAP_LABEL_FONT_SIZE, so at other sizes they are not expected to be enough.
sweep_rows = []
for font_size in LABEL_FONT_SIZES_TO_SWEEP:
    for nudges_name, nudges in [("no nudges", {}), ("with nudges", LABEL_OFFSET_POINTS_BY_WARD)]:
        sweep_figure, sweep_axis = plt.subplots(figsize=FIGURE_SIZE_MAP)
        ward_maps.draw_ward_outlines(sweep_axis, alder_map_table)
        sweep_annotations = ward_maps.draw_ward_labels(sweep_axis, alder_map_table, map_label_by_ward,
                                                       font_size=font_size, offset_points_by_ward=nudges)
        sweep_axis.set_axis_off()
        sweep_figure.canvas.draw()
        overlaps = ward_maps.label_overlaps(sweep_figure, sweep_annotations)
        plt.close(sweep_figure)
        sweep_rows.append({"font_size": font_size, "nudges": nudges_name, "overlapping_pairs": len(overlaps),
                           "pairs": ", ".join(f"{ward_a}/{ward_b}" for ward_a, ward_b, *_ in overlaps)})
display(pd.DataFrame(sweep_rows))

### 1.3 Basemap options side by side

The same map with each entry of `ward_maps.BASEMAP_OPTIONS`. Pick one for `BASEMAP_CHOICE`
in the settings cell. Panels are smaller than §1.2, so labels are 5 pt with no nudges.

In [ ]:
figure, axes = plt.subplots(1, len(ward_maps.BASEMAP_OPTIONS), figsize=FIGURE_SIZE_BASEMAP_COMPARISON)
for axis, basemap_option in zip(axes, ward_maps.BASEMAP_OPTIONS):
    ward_maps.draw_ward_outlines(axis, alder_map_table)
    ward_maps.draw_ward_labels(axis, alder_map_table, dict(zip(alder_map_table.ward, alder_map_table.map_label)),
                               font_size=5)
    ward_maps.add_basemap(axis, basemap_option)
    axis.set_title(f'BASEMAP_CHOICE = "{basemap_option}"')
    axis.set_axis_off()
plt.tight_layout()
plt.show()

### 1.4 Photo grid

Photos from `data/raw/photos/` (`scripts/fetch_photos.py`, from `people.photo_url`, saved unchanged).
The manifest table shows each file's source, retrieval time, size and pixel dimensions. The files
differ a lot in resolution. The grid draws each photo at the same cell size; non-square photos are
cropped for display per `PHOTO_DISPLAY_CROP` / `PHOTO_CROP_VERTICAL_ANCHOR` (files unchanged).

In [ ]:
photo_manifest = pd.DataFrame(json.loads((PHOTOS_DIR / "manifest.json").read_text())).T
photo_manifest.index.name = "person_id"
photo_manifest["pixel_width_height"] = [Image.open(PHOTOS_DIR / file_name).size for file_name in photo_manifest.file]

# Every current alder has exactly one photo, and no extra photos.
assert set(photo_manifest.index) == set(alder_map_table.person_id), "photo set differs from current alders"
photo_manifest = photo_manifest.sort_values("elms_ward")
display(photo_manifest[["elms_ward", "display_name", "file", "content_type", "bytes", "pixel_width_height", "retrieved_at"]])

In [ ]:
# Every non-square photo: as stored, and both crop anchors, so PHOTO_CROP_VERTICAL_ANCHOR can be checked.
non_square_photos = photo_manifest[[width != height for width, height in photo_manifest.pixel_width_height]]
print(f"non-square photos: {len(non_square_photos)}")
crop_versions = ["as stored", "top", "center"]
figure, axes_grid = plt.subplots(len(non_square_photos), len(crop_versions), figsize=(7, 3 * len(non_square_photos)),
                                 squeeze=False)
for axes_row, photo in zip(axes_grid, non_square_photos.itertuples()):
    image = Image.open(PHOTOS_DIR / photo.file)
    for axis, crop_version in zip(axes_row, crop_versions):
        shown = image if crop_version == "as stored" else ward_maps.crop_to_square(image, crop_version)
        axis.imshow(shown)
        axis.set_title(f"{photo.display_name}\n{crop_version} {shown.size}", fontsize=8)
        axis.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
def photo_for_display(person_id):
    image = Image.open(PHOTOS_DIR / photo_manifest.loc[person_id, "file"])
    if PHOTO_DISPLAY_CROP == "square":
        return ward_maps.crop_to_square(image, PHOTO_CROP_VERTICAL_ANCHOR)
    if PHOTO_DISPLAY_CROP == "none":
        return image
    raise ValueError(PHOTO_DISPLAY_CROP)


row_count = -(-len(alder_map_table) // PHOTO_GRID_COLUMNS)   # ceiling division
figure, axes_grid = plt.subplots(row_count, PHOTO_GRID_COLUMNS, figsize=(2 * PHOTO_GRID_COLUMNS, 2.6 * row_count))
for axis, alder in zip(axes_grid.flat, alder_map_table.itertuples()):
    axis.imshow(photo_for_display(alder.person_id))
    axis.set_title(f"{alder.ward.lstrip('0')} — {alder.label_name}", fontsize=9)
    axis.set_axis_off()
for unused_axis in list(axes_grid.flat)[len(alder_map_table):]:
    unused_axis.set_axis_off()
plt.tight_layout()
plt.show()

## 2. Tenure map

### 2.1 Years on council — the table the map draws

`people.years_on_council` as built by `scripts/build_tables.py` (decision 1.2): from the earliest
DataMade City Council membership start (gaps ignored) to the earlier of the last DataMade
membership end and `AS_OF_DATE`, in days / 365.25. Shown here unrounded, with the columns it comes from.
Note: ward 27's Walter R. Burnett was appointed 2025-09-25 to succeed his father, Walter Burnett Jr.
They are two people in both eLMS and DataMade (sessions/2026-10-04.md, decision 1.2).

In [ ]:
tenure_table = alder_map_table[["ward", "person_id", "display_name", "label_name", "council_start_date",
                                "tenure_end_date", "start_source", "datamade_membership_count", "years_on_council"]]
display(tenure_table.sort_values(["years_on_council", "ward"], ascending=[False, True]).reset_index(drop=True))
print("years_on_council summary:", tenure_table.years_on_council.describe().round(2).to_dict())
print("alders per distinct value (shared term starts):")
display(tenure_table.years_on_council.round(2).value_counts().sort_index().rename("alders").to_frame())

### 2.2 Map — wards colored by years on council

Continuous `TENURE_COLORMAP` from `TENURE_COLOR_MIN_RULE` up to `TENURE_COLOR_CAP_YEARS` (values above
the cap get the top color; the color bar's arrow marks that), fill opacity `TENURE_FILL_OPACITY`, basemap
`BASEMAP_CHOICE`. Both label styles are drawn, the one chosen in `TENURE_LABEL_STYLE` first. Each
map lists its overlapping label pairs (expected: none), and a font-size sweep follows.

In [ ]:
if TENURE_COLOR_MIN_RULE == "data_min":
    tenure_color_min = tenure_table.years_on_council.min()
elif TENURE_COLOR_MIN_RULE == "zero":
    tenure_color_min = 0
else:
    raise ValueError(TENURE_COLOR_MIN_RULE)
tenure_color_max = TENURE_COLOR_CAP_YEARS if TENURE_COLOR_CAP_YEARS is not None else tenure_table.years_on_council.max()
print(f"color scale: {tenure_color_min:.2f} -> {tenure_color_max:.2f} years "
      f"(TENURE_COLOR_MIN_RULE={TENURE_COLOR_MIN_RULE!r}, TENURE_COLOR_CAP_YEARS={TENURE_COLOR_CAP_YEARS})")
print("alders above the top of the scale (shown in the top color):")
display(tenure_table[tenure_table.years_on_council > tenure_color_max][["ward", "display_name", "years_on_council"]])

years_text_by_ward = {ward: f"{years:.{TENURE_DECIMALS}f}"
                      for ward, years in zip(alder_map_table.ward, alder_map_table.years_on_council)}
number_name_by_ward = {ward: f"{ward.lstrip('0')} {name}" for ward, name in zip(alder_map_table.ward, alder_map_table.label_name)}


def draw_tenure_labels(axis, label_style, offsets, three_lines_font_size=TENURE_THREE_LINES_FONT_SIZE,
                       years_font_size=TENURE_YEARS_FONT_SIZE):
    """Labels for one TENURE_LABEL_STYLE option. Returns what ward_maps.label_overlaps needs."""
    if label_style == "three_lines":
        three_line_text = {ward: f"{map_label_by_ward[ward]}\n{years_text_by_ward[ward]}" for ward in map_label_by_ward}
        return ward_maps.draw_ward_labels(axis, alder_map_table, three_line_text, font_size=three_lines_font_size,
                                          offset_points_by_ward=offsets)
    if label_style == "two_sizes":
        return ward_maps.draw_ward_two_size_labels(axis, alder_map_table, number_name_by_ward, years_text_by_ward,
                                                   TENURE_NAME_FONT_SIZE, years_font_size, offset_points_by_ward=offsets)
    raise ValueError(label_style)


label_styles_in_order = [TENURE_LABEL_STYLE] + [style for style in TENURE_LABEL_OFFSETS_BY_STYLE if style != TENURE_LABEL_STYLE]
for label_style in label_styles_in_order:
    figure, axis = plt.subplots(figsize=FIGURE_SIZE_MAP)
    color_mappable = ward_maps.draw_ward_fill(axis, alder_map_table, "years_on_council", TENURE_COLORMAP,
                                              tenure_color_min, tenure_color_max, TENURE_FILL_OPACITY)
    ward_maps.draw_ward_outlines(axis, alder_map_table)
    tenure_annotations = draw_tenure_labels(axis, label_style, TENURE_LABEL_OFFSETS_BY_STYLE[label_style])
    ward_maps.add_basemap(axis, BASEMAP_CHOICE)
    # Color bar inside the map's empty lower-left corner, so it does not shrink the map (nudges assume FIGURE_SIZE_MAP).
    color_bar_axis = axis.inset_axes([0.04, 0.06, 0.025, 0.32])
    above_cap = (tenure_table.years_on_council > tenure_color_max).any()
    color_bar = figure.colorbar(color_mappable, cax=color_bar_axis, alpha=TENURE_FILL_OPACITY,
                                extend="max" if above_cap else "neither")   # arrow = values above the top
    color_bar.set_label("years on council")
    color_bar_ticks = sorted({round(tenure_color_min, 2), round(tenure_color_max, 2)}
                             | {tick for tick in (5, 10, 15, 20, 25) if tenure_color_min < tick < tenure_color_max})
    color_bar.set_ticks(color_bar_ticks, labels=[f"{tick:g}" for tick in color_bar_ticks])
    chosen_note = "TENURE_LABEL_STYLE (chosen)" if label_style == TENURE_LABEL_STYLE else "other option"
    axis.set_title(f"Years on council, current alders (people.years_on_council, as of AS_OF_DATE)\n"
                   f'label style "{label_style}" — {chosen_note}')
    axis.set_axis_off()
    figure.canvas.draw()
    tenure_overlaps = ward_maps.label_overlaps(figure, tenure_annotations)
    plt.show()
    print(f'"{label_style}": overlapping label pairs: {len(tenure_overlaps)}',
          [f"{ward_a}/{ward_b}" for ward_a, ward_b, *_ in tenure_overlaps])

In [ ]:
# Font-size sweep for both label styles: overlapping pairs without and with each style's nudges
# (FIGURE_SIZE_MAP, outlines only; fill and basemap do not move labels).
def count_tenure_overlaps(label_style, offsets, **font_sizes):
    sweep_figure, sweep_axis = plt.subplots(figsize=FIGURE_SIZE_MAP)
    ward_maps.draw_ward_outlines(sweep_axis, alder_map_table)
    sweep_annotations = draw_tenure_labels(sweep_axis, label_style, offsets, **font_sizes)
    sweep_axis.set_axis_off()
    sweep_figure.canvas.draw()
    overlaps = ward_maps.label_overlaps(sweep_figure, sweep_annotations)
    plt.close(sweep_figure)
    return overlaps

tenure_sweep_rows = []
sweep_cases = ([("three_lines", {"three_lines_font_size": size}, f"all lines {size} pt")
                for size in TENURE_THREE_LINES_SIZES_TO_SWEEP]
               + [("two_sizes", {"years_font_size": size}, f"name {TENURE_NAME_FONT_SIZE} pt, years {size} pt")
                  for size in TENURE_YEARS_SIZES_TO_SWEEP])
for label_style, font_sizes, font_sizes_text in sweep_cases:
    for nudges_name, nudges in [("no nudges", {}), ("with nudges", TENURE_LABEL_OFFSETS_BY_STYLE[label_style])]:
        overlaps = count_tenure_overlaps(label_style, nudges, **font_sizes)
        tenure_sweep_rows.append({"label_style": label_style, "font_sizes": font_sizes_text, "nudges": nudges_name,
                                  "overlapping_pairs": len(overlaps),
                                  "pairs": ", ".join(f"{ward_a}/{ward_b}" for ward_a, ward_b, *_ in overlaps)})
display(pd.DataFrame(tenure_sweep_rows))

### 2.3 Sorted bar chart

Same values, colors and color range as the map (bars above `TENURE_COLOR_CAP_YEARS` get the top color;
bar lengths are the true values). Longest tenure at the top; equal values are
ordered by ward number.

In [ ]:
bar_order = tenure_table.sort_values(["years_on_council", "ward"], ascending=[True, False])   # barh draws bottom-up
bar_colors = [color_mappable.to_rgba(years) for years in bar_order.years_on_council]

figure, axis = plt.subplots(figsize=FIGURE_SIZE_BAR_CHART)
axis.barh([f"{ward.lstrip('0')} — {name}" for ward, name in zip(bar_order.ward, bar_order.label_name)],
          bar_order.years_on_council, height=TENURE_BAR_HEIGHT, color=bar_colors, alpha=TENURE_FILL_OPACITY)
for position, years in enumerate(bar_order.years_on_council):
    axis.annotate(f"{years:.{TENURE_DECIMALS}f}", (years, position), xytext=(3, 0), textcoords="offset points",
                  va="center", fontsize=TENURE_BAR_FONT_SIZE)
axis.set_xlabel(f"years on council (to tenure_end_date: {', '.join(sorted(tenure_table.tenure_end_date.unique()))})")
axis.set_ylim(-0.6, len(bar_order) - 0.4)
axis.tick_params(axis="y", labelsize=TENURE_BAR_FONT_SIZE)
axis.spines[["top", "right"]].set_visible(False)
axis.set_title("Years on council, current alders")
plt.tight_layout()
plt.show()

## 3. Absence map

### 3.1 Which events count, and the one left out

Absent from a meeting = vote "Absent" on more than `ABSENT_MEETING_SHARE_THRESHOLD` of that
meeting's counted items (decision 1.4). Counted items: `vote_events` with `roster_kind` in
`ABSENCE_ROSTER_KINDS` at meetings whose `body` is `ABSENCE_MEETING_BODY`, minus
`EXCLUDE_EVENTS_FROM_ABSENCE`. Same computation as `tables_review` §2.6. First, the evidence for the
excluded event: its roster against the current council and the meeting's attendance roll call.

In [ ]:
meetings_table = pd.read_csv(TABLES_DIR / "meetings.csv")
vote_events_table = pd.read_csv(TABLES_DIR / "vote_events.csv", low_memory=False)
member_votes_table = pd.read_csv(TABLES_DIR / "member_votes.csv")
attendance_table = pd.read_csv(TABLES_DIR / "attendance.csv")

for excluded_event_id, reason in EXCLUDE_EVENTS_FROM_ABSENCE.items():
    print("EXCLUDED:", reason)
    excluded_event = vote_events_table[vote_events_table.event_id == excluded_event_id]
    display(excluded_event[["record_number", "title", "action_date", "action_by", "roster_kind", "agreed_calendar",
                            "count_yea", "count_absent", "count_not_voting", "meeting_id"]])
    excluded_roster = (member_votes_table[member_votes_table.event_id == excluded_event_id]
                       .merge(people_table[["person_id", "display_name", "elms_ward", "is_active", "tenure_end_date"]],
                              on="person_id", how="left"))
    print("roster people who are not current alders:")
    display(excluded_roster[~excluded_roster.is_active.fillna(False).astype(bool)]
            [["display_name", "elms_ward", "tenure_end_date", "vote"]])
    print("votes other than Yea on this event:")
    display(excluded_roster[excluded_roster.vote != "Yea"][["display_name", "is_active", "vote"]])
    meeting_id = excluded_event.meeting_id.iloc[0]
    print("same meeting — all vote events and their roster kinds:")
    display(vote_events_table[vote_events_table.meeting_id == meeting_id]
            [["record_number", "action_date", "roster_kind", "roster_signature"]])
    print("same meeting — attendance roll call statuses:",
          attendance_table[attendance_table.meeting_id == meeting_id].status.value_counts().to_dict())

In [ ]:
council_meeting_ids = set(meetings_table.loc[meetings_table.body == ABSENCE_MEETING_BODY, "meeting_id"])
counted_events = vote_events_table[vote_events_table.roster_kind.isin(ABSENCE_ROSTER_KINDS)
                                   & vote_events_table.meeting_id.isin(council_meeting_ids)
                                   & ~vote_events_table.event_id.isin(EXCLUDE_EVENTS_FROM_ABSENCE)]
print(f"counted events: {len(counted_events)} at {counted_events.meeting_id.nunique()} meetings "
      f"(excluded: {len(EXCLUDE_EVENTS_FROM_ABSENCE)})")

# One row per alder x meeting: share of that meeting's counted items marked Absent.
alder_meeting = (member_votes_table.merge(counted_events[["event_id", "meeting_id"]], on="event_id")
                 .assign(is_absent_vote=lambda frame: frame.vote == "Absent")
                 .groupby(["meeting_id", "person_id"])
                 .agg(items=("is_absent_vote", "size"), absent_items=("is_absent_vote", "sum"))
                 .reset_index())
alder_meeting["share_of_items_absent"] = alder_meeting.absent_items / alder_meeting["items"]
alder_meeting["absent_from_meeting"] = alder_meeting.share_of_items_absent > ABSENT_MEETING_SHARE_THRESHOLD
alder_meeting["meeting_date"] = alder_meeting.meeting_id.map(meetings_table.set_index("meeting_id").meeting_date)
alder_meeting = alder_meeting[alder_meeting.person_id.isin(alder_map_table.person_id)]   # current alders only

print("roll-call items per meeting (one value per meeting):")
display(counted_events.groupby("meeting_id").size().describe().round(1).to_frame("items").T)
print("current alders' meetings on roster, by count:", alder_meeting.groupby("person_id").size().value_counts().to_dict())

# Cross-check (not used by the map): vote-based absence vs the meeting's attendance roll call.
cross_check = alder_meeting.merge(attendance_table[["meeting_id", "person_id", "status"]], on=["meeting_id", "person_id"], how="left")
print("absent_from_meeting (rows) x attendance roll call status (columns):")
display(pd.crosstab(cross_check.absent_from_meeting, cross_check.status.fillna("no attendance row"), margins=True))

### 3.2 Ranked table

Denominator: `ABSENCE_DENOMINATOR`. Sorted by percent absent, then ward. `absent_meeting_dates`
lists every meeting counted as absent.

In [ ]:
if ABSENCE_DENOMINATOR == "meetings_on_roster":
    denominator_by_person = alder_meeting.groupby("person_id").size()
elif ABSENCE_DENOMINATOR == "all_meetings":
    denominator_by_person = pd.Series(counted_events.meeting_id.nunique(), index=alder_map_table.person_id)
else:
    raise ValueError(ABSENCE_DENOMINATOR)

absence_table = alder_map_table[["ward", "person_id", "display_name", "label_name"]].copy()
absence_table["meetings_in_denominator"] = absence_table.person_id.map(denominator_by_person).fillna(0).astype(int)
absence_table["absent_meetings"] = (absence_table.person_id
                                    .map(alder_meeting.groupby("person_id").absent_from_meeting.sum()).fillna(0).astype(int))
absence_table["percent_absent"] = 100 * absence_table.absent_meetings / absence_table.meetings_in_denominator
absence_table["absent_meeting_dates"] = absence_table.person_id.map(
    alder_meeting[alder_meeting.absent_from_meeting].groupby("person_id").meeting_date
    .apply(lambda dates: ", ".join(sorted(dates)))).fillna("")
display(absence_table.sort_values(["percent_absent", "ward"], ascending=[False, True]).reset_index(drop=True)
        .drop(columns=["person_id", "label_name"]).round({"percent_absent": 2}))
print("alders by number of absent meetings:", absence_table.absent_meetings.value_counts().sort_index().to_dict())

### 3.3 Map — percent of meetings absent

`ABSENCE_COLORMAP` from `ABSENCE_COLOR_MIN_PERCENT` to `ABSENCE_COLOR_CAP_PERCENT` (None = highest),
basemap `BASEMAP_CHOICE`. Both label value options are drawn, `ABSENCE_LABEL_VALUE` first, each with its
overlapping label pairs (expected: none).

In [ ]:
absence_color_max = (ABSENCE_COLOR_CAP_PERCENT if ABSENCE_COLOR_CAP_PERCENT is not None
                     else absence_table.percent_absent.max())
print(f"color scale: {ABSENCE_COLOR_MIN_PERCENT} -> {absence_color_max:.2f} %")
absence_map_table = alder_map_table[["ward", "geometry"]].merge(absence_table, on="ward")

absence_value_text = {
    "both": {row.ward: f"{row.percent_absent:.{ABSENCE_PERCENT_DECIMALS}f}% ({row.absent_meetings}/{row.meetings_in_denominator})"
             for row in absence_table.itertuples()},
    "percent": {row.ward: f"{row.percent_absent:.{ABSENCE_PERCENT_DECIMALS}f}%" for row in absence_table.itertuples()},
}


def draw_absence_map(axis, figure, colormap_name, label_value):
    """Fill, outlines, optional labels, basemap and color bar. Returns the label annotations (or {})."""
    color_mappable = ward_maps.draw_ward_fill(axis, absence_map_table, "percent_absent", colormap_name,
                                              ABSENCE_COLOR_MIN_PERCENT, absence_color_max, ABSENCE_FILL_OPACITY)
    ward_maps.draw_ward_outlines(axis, absence_map_table)
    annotations = {}
    if label_value is not None:
        annotations = ward_maps.draw_ward_two_size_labels(
            axis, absence_map_table, number_name_by_ward, absence_value_text[label_value],
            ABSENCE_NAME_FONT_SIZE, ABSENCE_VALUE_FONT_SIZE, offset_points_by_ward=ABSENCE_LABEL_OFFSETS_BY_VALUE[label_value])
    ward_maps.add_basemap(axis, BASEMAP_CHOICE)
    above_cap = (absence_table.percent_absent > absence_color_max).any()
    color_bar = figure.colorbar(color_mappable, cax=axis.inset_axes([0.04, 0.06, 0.025, 0.32]),
                                alpha=ABSENCE_FILL_OPACITY, extend="max" if above_cap else "neither")
    color_bar.set_label("% of council meetings absent")
    axis.set_axis_off()
    return annotations


label_values_in_order = [ABSENCE_LABEL_VALUE] + [value for value in ABSENCE_LABEL_OFFSETS_BY_VALUE if value != ABSENCE_LABEL_VALUE]
for label_value in label_values_in_order:
    figure, axis = plt.subplots(figsize=FIGURE_SIZE_MAP)
    absence_annotations = draw_absence_map(axis, figure, ABSENCE_COLORMAP, label_value)
    chosen_note = "ABSENCE_LABEL_VALUE (chosen)" if label_value == ABSENCE_LABEL_VALUE else "other option"
    axis.set_title(f"% of council meetings absent (> {ABSENT_MEETING_SHARE_THRESHOLD:.0%} of roll-call items Absent), "
                   f"current alders\nlabel value \"{label_value}\" — {chosen_note}; colormap {ABSENCE_COLORMAP}")
    figure.canvas.draw()
    absence_overlaps = ward_maps.label_overlaps(figure, absence_annotations)
    plt.show()
    print(f'"{label_value}": overlapping label pairs: {len(absence_overlaps)}',
          [f"{ward_a}/{ward_b}" for ward_a, ward_b, *_ in absence_overlaps])

### 3.4 Colormap options side by side

Each entry of `ABSENCE_COLORMAP_OPTIONS`, same scale, no labels. Pick one for `ABSENCE_COLORMAP`.

In [ ]:
figure, axes = plt.subplots(1, len(ABSENCE_COLORMAP_OPTIONS), figsize=FIGURE_SIZE_COLORMAP_COMPARISON)
for axis, colormap_name in zip(axes, ABSENCE_COLORMAP_OPTIONS):
    draw_absence_map(axis, figure, colormap_name, label_value=None)
    axis.set_title(f'ABSENCE_COLORMAP = "{colormap_name}"')
plt.tight_layout()
plt.show()